# Getting Started with Claude Sonnet 5 on Amazon Bedrock

**Anthropic's most capable Sonnet model — delivering near-Opus 4.8 intelligence at Sonnet pricing, with strong reasoning, coding, and agentic reliability at scale.**

Claude Sonnet 5 is the model you recommend when a customer needs strong reasoning, coding, and agentic reliability at scale without the Opus-tier cost. It covers the majority of production workloads and is a drop-in upgrade from Sonnet 4.6.

---

## What's new in Claude Sonnet 5

- **Near-Opus intelligence at Sonnet cost**: Strong performance on SWE-bench, coding tasks, and knowledge work — at a fraction of Opus pricing.
- **Agentic reliability**: Reliable tool use across multi-step workflows. Finds paths around obstacles, recovers from errors, knows when to ask for help vs. keep going.
- **Adaptive thinking by default**: Extended reasoning is enabled out of the box with `high` as the default effort level — no configuration needed for most workloads.
- **Enterprise consistency**: Lower output variance, stronger instruction-following. Produces outputs teams can ship without re-checking.
- **Deep knowledge work**: Synthesizes across sources, makes sensible assumptions on underspecified requests, and self-checks output to get things right on the first pass.

---

## When to use Claude Sonnet 5

| Use case | Description |
|----------|-------------|
| **Agentic coding** | Feature work, migrations, refactors, and debugging in real codebases |
| **Research & analysis** | Synthesizing long, complex sources into briefs, analyses, and structured deliverables |
| **Agents & automation** | Customer-facing and internal agents that use tools reliably and run multi-step workflows |
| **Financial services** | Investment research, earnings analysis, credit/risk review, compliance workflows |
| **Legal** | Contract review, due diligence, case research, first drafts of motions and memos |
| **Life sciences** | Literature review, clinical documentation, regulatory submission drafting |
| **Cybersecurity** | Threat intelligence, vulnerability finding, alert triage, incident response, security code review |
| **High-volume production** | Workloads where Opus-tier cost is prohibitive but strong reasoning is required |

---

## Key capabilities

| Capability | Details |
|------------|---------|
| **Context window** | 1M tokens (native — no beta header or model-ID suffix required) |
| **Max output** | Up to 128K tokens |
| **Adaptive thinking** | Enabled by default with `high` effort. Extended reasoning that activates when needed. |
| **Task budgets** | Token spending limits for agentic loops (beta) |
| **Context compaction** | Automatic summarization for infinite context (beta) |
| **Vision** | High-resolution image input for screenshots, diagrams, charts, and design files |
| **Knowledge cutoff** | January 2026 |
| **Supported languages** | English, French, Arabic, Mandarin Chinese, Hindi, Spanish, Portuguese, Korean, Japanese, German, Russian, Polish, and others |

---

## Breaking changes from Sonnet 4.6

| Change | Impact |
|--------|--------|
| **Extended thinking deprecated** | `thinking: {"type": "enabled", "budget_tokens": N}` returns a **400 error**. Migrate to adaptive thinking (see below). |
| **Sampling parameters not supported** | `temperature`, `top_p`, or `top_k` set to any non-default value returns a **400 error**. Remove them from all calls. |
| **Model ID change** | Update model IDs (see table below). |

---

## Migrating from Sonnet 4.6

Sonnet 5 is a drop-in upgrade from Sonnet 4.6. The API surface is identical to the latest Opus models — adaptive thinking, compaction, task budgets, and effort levels all work the same way.

### Step-by-step

1. **Update model IDs:**
   - Messages API (Mantle): `anthropic.claude-sonnet-4-6` → `anthropic.claude-sonnet-5`
   - InvokeModel / Converse (Runtime): `us.anthropic.claude-sonnet-4-6` → `us.anthropic.claude-sonnet-5`

2. **Update IAM policy** if using `bedrock-mantle:Model` condition key

3. **Replace extended thinking with adaptive thinking:**

   Old format (Sonnet 4.6):
   ```json
   "thinking": {"type": "enabled", "budget_tokens": 10000}
   ```

   New format (Sonnet 5):
   ```json
   "thinking": {"type": "adaptive"},
   "output_config": {"effort": "high"}
   ```

   To disable thinking entirely:
   ```json
   "thinking": {"type": "disabled"}
   ```

4. **Remove sampling parameters** — omit `temperature`, `top_p`, and `top_k` entirely from requests

5. **Test with representative workloads** — Sonnet 5 may produce different (better) outputs on the same prompts



---

## API options on Amazon Bedrock

Claude Sonnet 5 is available through **Amazon Bedrock Mantle**:

| API | Model ID | When to use |
|-----|----------|-------------|
| **Messages API (Mantle)** | `anthropic.claude-sonnet-5` | Recommended. Full feature support via Anthropic's native format. |
| **InvokeModel API** | `us.anthropic.claude-sonnet-5` | Bedrock native API with Anthropic request body. Supports compaction. |
| **Converse API** | `us.anthropic.claude-sonnet-5` | Bedrock unified conversational API. Simplest integration. |

Additional inference profiles:

| Profile | Model ID |
|---------|----------|
| US cross-region (recommended) | `us.anthropic.claude-sonnet-5` |
| EU cross-region | `eu.anthropic.claude-sonnet-5` |
| Global | `global.anthropic.claude-sonnet-5` |

The **Messages API via Bedrock Mantle** is the recommended starting point for new integrations.

---

## 1. Setup and configuration

In [ ]:
# Install required packages (uncomment if needed)
#!pip install boto3 --upgrade 2>&1 | tail -3 
#!pip install "anthropic[bedrock]" --upgrade 2>&1 | tail -3 

In [ ]:
import boto3
import json
import base64
from botocore.config import Config
import re

# Try importing Anthropic Bedrock Mantle client
try:
    from anthropic import AnthropicBedrockMantle
    MANTLE_AVAILABLE = True
except ImportError:
    print(
        "Note: anthropic[bedrock] not installed. "
        "Messages API examples will be skipped. "
        "Install with: pip install anthropic[bedrock]"
    )
    MANTLE_AVAILABLE = False

# Configuration
REGION = "us-east-1"

# Mantle (Messages API) - recommended
MANTLE_MODEL_ID = "anthropic.claude-sonnet-5"
MANTLE_BASE_URL = f"https://bedrock-mantle.{REGION}.api.aws"

# Runtime (Converse / InvokeModel APIs)
RUNTIME_MODEL_ID = "us.anthropic.claude-sonnet-5"

# Longer timeout for thinking/compaction responses
FAST_CONFIG = Config(read_timeout=1200, retries={"max_attempts": 1})

# Initialize clients
session = boto3.Session()
bedrock_runtime = session.client(
    service_name="bedrock-runtime",
    region_name=REGION,
    config=FAST_CONFIG,
)

if MANTLE_AVAILABLE:
    mantle_client = AnthropicBedrockMantle(
        aws_region=REGION,
        base_url=f"{MANTLE_BASE_URL}/anthropic",
    )

print(f"Region:          {REGION}")
print(f"Mantle Model:    {MANTLE_MODEL_ID}")
print(f"Runtime Model:   {RUNTIME_MODEL_ID}")
print(f"Mantle Base URL: {MANTLE_BASE_URL}")
print(f"Mantle SDK:      {'available' if MANTLE_AVAILABLE else 'not installed'}")
print("Setup complete.")

---

## 2. Basic usage with Messages API (Bedrock Mantle)

The Messages API is the **recommended way** to use Claude Sonnet 5 on Amazon Bedrock. It uses Anthropic's native request/response format, authenticated with AWS SigV4 via the `anthropic[bedrock]` SDK.

**Important:** Do NOT include `temperature`, `top_p`, or `top_k` in your requests — Sonnet 5 does not support non-default sampling parameters and will return a 400 error.

In [ ]:
# Messages API - basic request
if MANTLE_AVAILABLE:
    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=2048,
            messages=[
                {
                    "role": "user",
                    "content": (
                        "What are three key considerations when designing "
                        "a distributed system for high availability?"
                    ),
                }
            ],
        )

        print(message.content[0].text)
        print(
            f"\nUsage: Input={message.usage.input_tokens}, "
            f"Output={message.usage.output_tokens}"
        )

    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping - install anthropic[bedrock] for Messages API support")

---

## 3. Adaptive thinking

Adaptive thinking gives Claude the freedom to reason deeply when a task requires it, and respond quickly when it doesn't. **In Sonnet 5, adaptive thinking is enabled by default with `high` effort** — you don't need to configure anything for most workloads.

**How it works:**
- `thinking.type` is `"adaptive"` by default — Claude decides whether to think based on task complexity
- Control depth with `output_config.effort` — higher effort = deeper reasoning on complex tasks
- Default effort is `high` — override only if you need speed (`low`/`medium`) or maximum depth (`xhigh`)
- By default, thinking content is **omitted** (empty string with a cryptographic signature for multi-turn verification). You can set `display: "summarized"` to receive a human-readable summary of Claude's reasoning process.

**Important:** Do NOT use `thinking.type: "enabled"` — extended thinking is deprecated. It will return a 400 error. Use `"adaptive"` or `"disabled"`.

### Effort levels

| Effort | Behavior | Best for |
|--------|----------|----------|
| **max** | Exhaustive analysis, cost is secondary | Hardest problems requiring Opus-level depth |
| **xhigh** | Maximum reasoning depth | Most complex analytical workloads |
| **high** | Deep reasoning (API default for Sonnet 5) | Complex tasks |
| **medium** | Balanced — may skip thinking for simple queries | Mixed workloads (recommended starting point) |
| **low** | Minimal thinking, prioritizes speed | Simple queries, cost-sensitive |

**Recommendation:** While the API default is `high`, Anthropic recommends **starting at `medium`** for most workloads and moving to `high` for agentic or memory-heavy work. As a rough cross-model mapping: Sonnet 5 at `medium` is comparable in intelligence to Sonnet 4.6 at `high`, and Sonnet 5 at `high` is comparable to Sonnet 4.6 at `max`.

In [ ]:
# Adaptive thinking - basic example
if MANTLE_AVAILABLE:
    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=16000,
            thinking={"type": "adaptive", "display": "summarized"},
            extra_body={"output_config": {"effort": "high"}},
            messages=[
                {
                    "role": "user",
                    "content": (
                         "What is the sum of the first 30 prime numbers? Show your work."
                    ),
                }
            ],
        )

        # Check if thinking was triggered
        has_thinking = any(block.type == "thinking" for block in message.content)
        print(f"Claude decided to think: {has_thinking}\n")

        for block in message.content:
            if block.type == "thinking":
                sig_len = len(block.signature) if block.signature else 0
                if block.thinking:
                    # Display is "summarized" — print the thinking summary
                    print(f"[Thinking summary]\n{block.thinking}\n")
                else:
                    # Display is omitted — only signature available
                    print(f"[Thinking block] Signature: {sig_len} chars")
                    print("(Omitted thinking - signature proves reasoning occurred)\n")
            elif block.type == "text":
                text = block.text
                print(f"Response:\n{text[:1000]}..." if len(text) > 1000 else f"Response:\n{text}")

        print(
            f"\nUsage: Input={message.usage.input_tokens}, "
            f"Output={message.usage.output_tokens}"
        )

    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping - install anthropic[bedrock]")

### Comparing effort levels

Different effort levels produce different reasoning depths and costs. Use this to find the right balance for your workload.

**Note:** Since Sonnet 5 defaults to `high` effort, you only need to set effort explicitly when you want a different level:

In [ ]:
# Compare effort levels on the same prompt
def test_effort_level(effort_level: str, prompt: str):
    if not MANTLE_AVAILABLE:
        return {"effort": effort_level, "error": "SDK not installed"}

    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=12000,
            thinking={"type": "adaptive"},
            extra_body={"output_config": {"effort": effort_level}},
            messages=[{"role": "user", "content": prompt}],
        )

        has_thinking = any(b.type == "thinking" for b in message.content)
        response_len = sum(len(b.text) for b in message.content if b.type == "text")

        return {
            "effort": effort_level,
            "thought": has_thinking,
            "input_tokens": message.usage.input_tokens,
            "output_tokens": message.usage.output_tokens,
            "response_chars": response_len,
        }
    except Exception as e:
        return {"effort": effort_level, "error": str(e)}


prompt = (
    "Prove that there are infinitely many prime numbers. "
    "Then explain why this proof technique cannot be used to prove "
    "the twin prime conjecture."
)

print("Testing effort levels on the same prompt...\n")
print(f"{'Effort':<8} {'Thought':<8} {'Input':<8} {'Output':<8} {'Response':<10}")
print("-" * 50)

for effort in ["low", "medium", "high", "xhigh", "max"]:
    result = test_effort_level(effort, prompt)
    if "error" not in result:
        print(
            f"{result['effort']:<8} "
            f"{str(result['thought']):<8} "
            f"{result['input_tokens']:<8} "
            f"{result['output_tokens']:<8} "
            f"{result['response_chars']:<10}"
        )
    else:
        print(f"{result['effort']:<8} Error: {result['error']}")

print("\nHigher effort = deeper reasoning, more tokens, better quality on complex tasks.")

---

## 4. Streaming

Streaming delivers tokens as they are generated, reducing perceived latency for long responses. Each API has its own streaming mechanism.

In [ ]:
import sys

# Streaming with Messages API (Bedrock Mantle)
if MANTLE_AVAILABLE:
    print("Messages API streaming:\n")
    try:
        with mantle_client.messages.stream(
            model=MANTLE_MODEL_ID,
            max_tokens=512,
            messages=[{"role": "user", "content": "List five best practices for API security."}],
        ) as stream:
            for text in stream.text_stream:
                print(text, end="", flush=True)
        print()
    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping Messages API streaming - install anthropic[bedrock]")

# Streaming with InvokeModel API
print("\n\nInvokeModel API streaming:\n")
try:
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 512,
        "messages": [
            {"role": "user", "content": "List five best practices for container security."}
        ],
    }
    response = bedrock_runtime.invoke_model_with_response_stream(
        modelId=RUNTIME_MODEL_ID, body=json.dumps(request_body)
    )
    for event in response["body"]:
        chunk = json.loads(event["chunk"]["bytes"])
        if chunk["type"] == "content_block_delta":
            delta = chunk["delta"]
            if delta["type"] == "text_delta":
                print(delta["text"], end="", flush=True)
    print()
except Exception as e:
    print(f"Error: {e}")

# Streaming with Converse API
print("\n\nConverse API streaming:\n")
try:
    response = bedrock_runtime.converse_stream(
        modelId=RUNTIME_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": "List five best practices for IAM security."}]}],
        inferenceConfig={"maxTokens": 512},
    )
    for event in response["stream"]:
        if "contentBlockDelta" in event:
            delta = event["contentBlockDelta"].get("delta", {})
            if "text" in delta:
                print(delta["text"], end="", flush=True)
    print()
except Exception as e:
    print(f"Error: {e}")

---

## 5. Task budgets (beta)

Task budgets let you set token spending limits for entire agentic loops. This prevents runaway costs during autonomous operations while allowing Claude to complete complex multi-step tasks.

**How it works:**
- Set a maximum token budget via `output_config.task_budget`
- Claude tracks usage across the conversation/agentic loop
- When the budget is approached, Claude wraps up gracefully rather than stopping mid-task
- Ideal for autonomous agents, long-running workflows, and cost-controlled environments
- **Minimum budget is 20,000 tokens** — setting a value below this will return a 400 error

**Requires beta header:** `anthropic_beta: ["task-budgets-2026-03-13"]`

In [ ]:
# Task Budget example
if MANTLE_AVAILABLE:
    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=4096,
            thinking={"type": "adaptive"},
            extra_body={
                "anthropic_beta": ["task-budgets-2026-03-13"],
                "output_config": {
                    "effort": "high",
                    "task_budget": {"type": "tokens", "total": 25000},
                },
            },
            messages=[
                {
                    "role": "user",
                    "content": (
                        "Plan a comprehensive market research project for a new "
                        "fintech startup targeting small businesses. Include "
                        "competitor analysis, market sizing, and go-to-market "
                        "strategy recommendations."
                    ),
                }
            ],
        )

        print(f"Stop reason: {message.stop_reason}")
        print(f"Output tokens used: {message.usage.output_tokens} (budget: 25,000)")

        for block in message.content:
            if block.type == "text":
                text = block.text
                print(f"\nResponse:\n{text[:1000]}..." if len(text) > 1000 else f"\nResponse:\n{text}")

    except Exception as e:
        print(f"Error: {e}")
        print("Note: Task Budgets require the beta header and may not be available in all regions.")
else:
    print("Skipping - install anthropic[bedrock]")

---

## 6. Prompt caching

Prompt caching reduces latency and cost when you repeatedly send the same large context (system prompts, documents, examples) alongside different user queries.

**How it works:**
- Mark content as cacheable with `"cache_control": {"type": "ephemeral"}`
- The first request writes the cache; subsequent requests read from it
- Cache reads are significantly cheaper and faster than re-processing the same tokens
- Cache lifetime is 5 minutes (refreshed on each read)

**When to use it:**
- Large system prompts used across many requests
- Reference documents or codebases included with every query
- Few-shot examples that stay constant across a session

In [ ]:
# Prompt caching — Messages API
if MANTLE_AVAILABLE:
    # Simulate a large system prompt that would be reused across many requests
    large_system = "You are an expert software engineer.\n" + "\n".join(
        [f"Rule {i}: Always write clean, well-documented, production-ready code. "
         f"Follow best practices for error handling, testing, and performance. "
         f"Consider edge cases and security implications in every response."
         for i in range(1, 251)]
    )

    print(f"System prompt size: ~{len(large_system)} chars\n")

    try:
        # First request — populates the cache
        msg1 = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=64,
            system=[{
                "type": "text",
                "text": large_system,
                "cache_control": {"type": "ephemeral"},
            }],
            messages=[{"role": "user", "content": "What are Python anti-patterns to avoid?"}],
        )
        cache_write = getattr(msg1.usage, "cache_creation_input_tokens", 0) or 0
        cache_read_1 = getattr(msg1.usage, "cache_read_input_tokens", 0) or 0
        print(f"Request 1: cache_write={cache_write}, cache_read={cache_read_1}")

        # Second request — should hit the cache
        msg2 = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=64,
            system=[{
                "type": "text",
                "text": large_system,
                "cache_control": {"type": "ephemeral"},
            }],
            messages=[{"role": "user", "content": "How do I write effective unit tests?"}],
        )
        cache_read_2 = getattr(msg2.usage, "cache_read_input_tokens", 0) or 0
        print(f"Request 2: cache_read={cache_read_2}")

        if cache_read_2 > 0:
            print(f"\n✓ Cache hit! {cache_read_2} tokens served from cache.")
        elif cache_write > 0:
            print(f"\n✓ Cache written ({cache_write} tokens). Run again quickly to see a cache read.")

    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping - install anthropic[bedrock]")

# Prompt caching — InvokeModel API
print("\n" + "=" * 50)
print("\nPrompt caching with InvokeModel API:")

large_system_im = "You are a helpful coding assistant.\n" + "\n".join(
    [f"Rule {i}: Write clean code." for i in range(1, 251)]
)

request_body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 64,
    "system": [{
        "type": "text",
        "text": large_system_im,
        "cache_control": {"type": "ephemeral"},
    }],
    "messages": [{"role": "user", "content": "Say hello briefly."}],
}

try:
    # First call — cache write
    resp1 = bedrock_runtime.invoke_model(modelId=RUNTIME_MODEL_ID, body=json.dumps(request_body))
    body1 = json.loads(resp1["body"].read())
    usage1 = body1.get("usage", {})

    # Second call — cache read
    resp2 = bedrock_runtime.invoke_model(modelId=RUNTIME_MODEL_ID, body=json.dumps(request_body))
    body2 = json.loads(resp2["body"].read())
    usage2 = body2.get("usage", {})

    print(f"Request 1: cache_write={usage1.get('cache_creation_input_tokens', 0)}")
    print(f"Request 2: cache_read={usage2.get('cache_read_input_tokens', 0)}")
except Exception as e:
    print(f"Error: {e}")

---

## 7. Structured outputs (JSON schema)

Structured outputs (response_format: json_schema) is not currently supported on Amazon Bedrock — requests using it will return a 400 error. This is a known feature gap.

**Workarounds:**

- **JSON via prompting** — request JSON output with an example shape in the prompt. Sonnet 5 follows this reliably.
- **Output format via literal example** — include a concrete example of the desired format (e.g., "a coordinate pair (x, y), e.g. (123, 456)"). A single example is often sufficient.


In [ ]:
prompt = (
    "List exactly 3 programming languages with their year of "
    "creation. Respond with ONLY a JSON object, no prose and no "
    "markdown code fences, matching this shape exactly:\n"
    '{"languages": [{"name": "Python", "year": 1991}]}'
)

# JSON via prompting — Messages API
if MANTLE_AVAILABLE:
    print("Messages API — structured output:\n")
    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=256,
            messages=[{"role": "user", "content": prompt}]
        )
        text = message.content[0].text.strip()
        if text.startswith("```"):
            text = text.strip("`")
            text = text[text.find("{"):text.rfind("}") + 1]
        parsed = json.loads(text)
        print(json.dumps(parsed, indent=2))
    except Exception as e:
        print(f"Error: {e}")

# JSON via prompting — InvokeModel API
print("\n\nInvokeModel API — structured output:\n")
try:
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 256,
        "messages": [{"role": "user", "content": [{"type": "text", "text": prompt}]}],
    }
    response = bedrock_runtime.invoke_model(modelId=RUNTIME_MODEL_ID, body=json.dumps(request_body))
    body = json.loads(response["body"].read())
    text = body["content"][0]["text"].strip()
    if text.startswith("```"):
        text = text.strip("`")
        text = text[text.find("{"):text.rfind("}") + 1]
    parsed = json.loads(text)
    print(json.dumps(parsed, indent=2))
except Exception as e:
    print(f"Error: {e}")

# JSON via prompting — Converse API
print("\n\nConverse API — structured output:\n")
try:
    response = bedrock_runtime.converse(
        modelId=RUNTIME_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 256},
    )
    text = response["output"]["message"]["content"][0]["text"].strip()
    if text.startswith("```"):
        text = text.strip("`")
        text = text[text.find("{"):text.rfind("}") + 1]
    parsed = json.loads(text)
    print(json.dumps(parsed, indent=2))
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Output format via literal example — Messages API
# The prompt uses a literal example to enforce output format
prompt = (
    "What are the pixel coordinates of the exact center of a "
    "200 by 100 image? Return your answer as a coordinate pair "
    "(x, y), e.g. (123, 456). Output only the pair."
)

# Output format via literal example — Messages API
if MANTLE_AVAILABLE:
    print("Messages API — output format via literal example:\n")
    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=64,
            messages=[{"role": "user", "content": prompt}]
        )
        text = message.content[0].text.strip()

        # Validate the response matches the expected (x, y) format
        passed = bool(re.search(r"\(\s*\d+\s*,\s*\d+\s*\)", text))
        if passed:
            print(f"Matched (x, y): {text}")    # e.g. "(100, 50)"
        else:
            print(f"Did not match expected shape: {text}")
    except Exception as e:
        print(f"Error: {e}")

# Output format via literal example — InvokeModel API
print("\n\nInvokeModel API — output format via literal example:\n")
try:
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 64,
        "messages": [{"role": "user", "content": [{"type": "text", "text": prompt}]}],
    }
    response = bedrock_runtime.invoke_model(modelId=RUNTIME_MODEL_ID, body=json.dumps(request_body))
    body = json.loads(response["body"].read())
    text = body["content"][0]["text"].strip()

    passed = bool(re.search(r"\(\s*\d+\s*,\s*\d+\s*\)", text))
    if passed:
        print(f"Matched (x, y): {text}")
    else:
        print(f"Did not match expected shape: {text}")
except Exception as e:
    print(f"Error: {e}")

# Output format via literal example — Converse API
print("\n\nConverse API — output format via literal example:\n")
try:
    response = bedrock_runtime.converse(
        modelId=RUNTIME_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 64},
    )
    text = response["output"]["message"]["content"][0]["text"].strip()

    passed = bool(re.search(r"\(\s*\d+\s*,\s*\d+\s*\)", text))
    if passed:
        print(f"Matched (x, y): {text}")
    else:
        print(f"Did not match expected shape: {text}")
except Exception as e:
    print(f"Error: {e}")

---

## 8. Tool use (all 3 APIs)

Tool use lets Claude call functions you define. Each API has a slightly different format for defining and invoking tools.

In [ ]:
# Tool use — Messages API
if MANTLE_AVAILABLE:
    print("Messages API — tool use:\n")
    try:
        tools = [{
            "name": "calculator",
            "description": "Performs basic arithmetic. Use for any math question.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "The math expression to evaluate"}
                },
                "required": ["expression"],
            },
        }]
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=512,
            tools=tools,
            messages=[{"role": "user", "content": "What is 42 * 17? Use the calculator tool."}],
        )
        has_tool_use = any(b.type == "tool_use" for b in message.content)
        print(f"Tool used: {has_tool_use}, Stop reason: {message.stop_reason}")
        for block in message.content:
            if block.type == "tool_use":
                print(f"Tool: {block.name}, Input: {json.dumps(block.input)}")
    except Exception as e:
        print(f"Error: {e}")

# Tool use — InvokeModel API (same tool format as Messages API)
print("\n\nInvokeModel API — tool use:\n")
try:
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 512,
        "tools": [{
            "name": "calculator",
            "description": "Performs basic arithmetic.",
            "input_schema": {
                "type": "object",
                "properties": {"expression": {"type": "string"}},
                "required": ["expression"],
            },
        }],
        "messages": [{"role": "user", "content": "What is 42 * 17? Use the calculator tool."}],
    }
    response = bedrock_runtime.invoke_model(modelId=RUNTIME_MODEL_ID, body=json.dumps(request_body))
    body = json.loads(response["body"].read())
    has_tool_use = any(b["type"] == "tool_use" for b in body["content"])
    print(f"Tool used: {has_tool_use}, Stop reason: {body.get('stop_reason')}")
    for block in body["content"]:
        if block["type"] == "tool_use":
            print(f"Tool: {block['name']}, Input: {json.dumps(block['input'])}")
except Exception as e:
    print(f"Error: {e}")

# Tool use — Converse API (Bedrock toolConfig format)
print("\n\nConverse API — tool use:\n")
try:
    tool_config = {
        "tools": [{
            "toolSpec": {
                "name": "calculator",
                "description": "Performs basic arithmetic.",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {"expression": {"type": "string", "description": "Math expression"}},
                        "required": ["expression"],
                    }
                },
            }
        }]
    }
    response = bedrock_runtime.converse(
        modelId=RUNTIME_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": "What is 42 * 17? Use the calculator tool."}]}],
        inferenceConfig={"maxTokens": 512},
        toolConfig=tool_config,
    )
    content = response["output"]["message"]["content"]
    has_tool_use = any(block.get("toolUse") for block in content)
    print(f"Tool used: {has_tool_use}, Stop reason: {response.get('stopReason')}")
    for block in content:
        if block.get("toolUse"):
            tu = block["toolUse"]
            print(f"Tool: {tu['name']}, Input: {json.dumps(tu['input'])}")
except Exception as e:
    print(f"Error: {e}")

## 9. Context compaction (beta)

Context compaction enables "infinite context" for long-running conversations and agentic tasks by automatically summarizing older context when approaching the context window limit.

**How it works:**
1. Enable compaction with `context_management.edits` containing a `compact_20260112` edit
2. Set a `trigger` threshold (e.g. 100K tokens) — when input exceeds this, Claude summarizes older context
3. The API returns a `compaction` content block in the response
4. Pass the compaction block back in subsequent requests to maintain continuity

**API support:**

| API | Supported | Notes |
|-----|-----------|-------|
| **Messages API** | Yes | Via `extra_body` |
| **InvokeModel API** | Yes | In request body directly |
| **Converse API** | No | Returns a validation error |

**Requires beta header:** `anthropic_beta: ["compact-2026-01-12"]`

**Important notes:**
- Compaction won't trigger on short conversations — it's designed for 100K+ token contexts
- In multi-turn conversations, always preserve compaction blocks in the assistant response and pass them back
- Set the trigger value based on your use case (default recommendation: 100,000 tokens)

In [ ]:
# Context compaction - basic setup
if MANTLE_AVAILABLE:
    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=1024,
            extra_body={
                "anthropic_beta": ["compact-2026-01-12"],
                "context_management": {
                    "edits": [
                        {
                            "type": "compact_20260112",
                            "trigger": {
                                "type": "input_tokens",
                                "value": 100000,  # Trigger compaction at 100K tokens
                            },
                        }
                    ]
                },
            },
            messages=[
                {
                    "role": "user",
                    "content": (
                        "Let's start a long-running conversation about building "
                        "a complex e-commerce platform. What are the key "
                        "architectural decisions we need to make first?"
                    ),
                }
            ],
        )

        has_compaction = any(
            getattr(b, "type", None) == "compaction" for b in message.content
        )
        print(f"Compaction triggered: {has_compaction}")
        print("(Won't trigger on short conversations - designed for 100K+ token contexts)\n")

        for block in message.content:
            if block.type == "text":
                text = block.text
                print(f"{text[:800]}..." if len(text) > 800 else text)

        print(
            f"\nUsage: Input={message.usage.input_tokens}, "
            f"Output={message.usage.output_tokens}"
        )

    except Exception as e:
        print(f"Error: {e}")
        print("Note: Compaction requires the beta header.")
else:
    print("Skipping - install anthropic[bedrock]")

### Multi-turn conversation with compaction

In production, you pass compaction blocks back in the conversation history to maintain context across long sessions:

In [ ]:
# Multi-turn conversation with automatic compaction handling
if MANTLE_AVAILABLE:
    conversation = []

    def chat_with_compaction(user_message: str):
        """Send a message and handle compaction blocks automatically."""
        conversation.append({"role": "user", "content": user_message})

        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=4096,
            messages=conversation,
            extra_body={
                "anthropic_beta": ["compact-2026-01-12"],
                "context_management": {
                    "edits": [
                        {
                            "type": "compact_20260112",
                            "trigger": {"type": "input_tokens", "value": 100000},
                        }
                    ]
                },
            },
        )

        # Build assistant response preserving compaction blocks
        assistant_content = []
        response_text = ""
        compacted = False

        for block in message.content:
            if block.type == "text":
                assistant_content.append({"type": "text", "text": block.text})
                response_text += block.text
            elif block.type == "compaction":
                assistant_content.append(
                    {"type": "compaction", "compaction": block.compaction}
                )
                compacted = True

        conversation.append({"role": "assistant", "content": assistant_content})

        if compacted:
            print("[Compaction occurred - older context was summarized]\n")

        return response_text

    # Multi-turn demo
    try:
        print("Turn 1:")
        r1 = chat_with_compaction("Design the database schema for an e-commerce platform")
        print(f"{r1[:400]}...\n")
        print("=" * 50)

        print("\nTurn 2:")
        r2 = chat_with_compaction("Now add authentication and authorization")
        print(f"{r2[:400]}...\n")

        print(f"Conversation: {len(conversation)} messages")

    except Exception as e:
        print(f"Error: {e}")
else:
    print("Skipping - install anthropic[bedrock]")

---

## 10. High-resolution vision

Claude Sonnet 5 supports high-resolution image input, providing better accuracy on charts, dense documents, screenshots, and complex diagrams as compared to Sonnet 4.6 especially at higher effort on dense content. This is particularly useful for financial analysis (reading dense filings and charts) and cybersecurity (analyzing screenshots and logs).

In [ ]:
# Vision example
import struct, zlib

def create_sample_chart_png():
    """Create a simple 200x100 PNG with colored bars (simulates a bar chart)."""
    w, h = 200, 100
    bar_colors = [(66, 133, 244), (234, 67, 53), (251, 188, 4), (52, 168, 83)]
    bar_heights = [80, 55, 70, 90]
    bar_width, gap, start_x = 30, 15, 20
    raw = b''
    for y in range(h):
        raw += b'\x00'
        for x in range(w):
            drawn = False
            for i, (bh, color) in enumerate(zip(bar_heights, bar_colors)):
                bx = start_x + i * (bar_width + gap)
                if bx <= x < bx + bar_width and y >= h - bh:
                    raw += bytes(color)
                    drawn = True
                    break
            if not drawn:
                raw += b'\xff\xff\xff'
    def chunk(ct, d):
        c = ct + d
        return struct.pack('>I', len(d)) + c + struct.pack('>I', zlib.crc32(c) & 0xFFFFFFFF)
    ihdr = struct.pack('>IIBBBBB', w, h, 8, 2, 0, 0, 0)
    return (b'\x89PNG\r\n\x1a\n'
            + chunk(b'IHDR', ihdr)
            + chunk(b'IDAT', zlib.compress(raw))
            + chunk(b'IEND', b''))

sample_image = create_sample_chart_png()
image_b64 = base64.b64encode(sample_image).decode('utf-8')
print(f"Generated sample chart: {len(sample_image)} bytes")
print("(In production, use real high-res images to leverage Sonnet 5's enhanced vision)\n")

image_content = [
    {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_b64,
        },
    },
    {
        "type": "text",
        "text": "Describe this chart. What type is it? What do the colors represent?",
    },
]

# Vision — Messages API
if MANTLE_AVAILABLE:
    print("Messages API — vision:\n")
    try:
        message = mantle_client.messages.create(
            model=MANTLE_MODEL_ID,
            max_tokens=2048,
            messages=[{"role": "user", "content": image_content}],
        )
        print(f"{message.content[0].text}")
        print(f"\nUsage: Input={message.usage.input_tokens}, Output={message.usage.output_tokens}")
    except Exception as e:
        print(f"Error: {e}")

# Vision — InvokeModel API
print("\n\nInvokeModel API — vision:\n")
try:
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 2048,
        "messages": [{"role": "user", "content": image_content}],
    }
    response = bedrock_runtime.invoke_model(modelId=RUNTIME_MODEL_ID, body=json.dumps(request_body))
    body = json.loads(response["body"].read())
    print(f"{body['content'][0]['text']}")
    print(f"\nUsage: Input={body['usage']['input_tokens']}, Output={body['usage']['output_tokens']}")
except Exception as e:
    print(f"Error: {e}")

# Vision — Converse API
print("\n\nConverse API — vision:\n")
try:
    response = bedrock_runtime.converse(
        modelId=RUNTIME_MODEL_ID,
        messages=[{
            "role": "user",
            "content": [
                {
                    "image": {
                        "format": "png",
                        "source": {"bytes": sample_image},
                    }
                },
                {"text": "Describe this chart. What type is it? What do the colors represent?"},
            ],
        }],
        inferenceConfig={"maxTokens": 2048},
    )
    print(f"{response['output']['message']['content'][0]['text']}")
    print(f"\nUsage: Input={response['usage']['inputTokens']}, Output={response['usage']['outputTokens']}")
except Exception as e:
    print(f"Error: {e}")

---

## 11. Migration checklist

### From Sonnet 4.6

| Step | Action |
|------|--------|
| 1 | Replace model ID: `anthropic.claude-sonnet-4-6` → `anthropic.claude-sonnet-5` (Mantle) or `us.anthropic.claude-sonnet-4-6` → `us.anthropic.claude-sonnet-5` (Runtime) |
| 2 | Update IAM policy if using `bedrock-mantle:Model` condition key |
| 3 | Replace extended thinking: `{"type": "enabled", "budget_tokens": N}` → `{"type": "adaptive"}` with `"output_config": {"effort": "high"}` |
| 4 | Remove `temperature`, `top_p`, and `top_k` from all API calls |
| 5 | Test with representative workloads — outputs may differ (improve) |

### Example: before and after

**Sonnet 4.6 request:**
```json
{
  "model": "us.anthropic.claude-sonnet-4-6",
  "thinking": {"type": "enabled", "budget_tokens": 10000},
  "temperature": 0.7,
  "messages": [...]
}
```

**Sonnet 5 request:**
```json
{
  "model": "us.anthropic.claude-sonnet-5",
  "thinking": {"type": "adaptive"},
  "output_config": {"effort": "high"},
  "messages": [...]
}
```

---

## 12. Next steps

- **Start with defaults** — adaptive thinking at `high` effort works for most workloads out of the box
- **Tune effort levels** — drop to `medium` or `low` for simple/high-volume tasks to save cost; use `xhigh` for maximum reasoning depth
- **Enable compaction** for long-running agents to avoid context window limits
- **Set task budgets** for autonomous workflows to control costs
- **Try high-res vision** with real screenshots, charts, and documents from your domain
- **Build agentic loops** — Sonnet 5 excels at multi-step tool use with error recovery at scale

### Resources

- [Amazon Bedrock documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/)
- [Anthropic SDK for Bedrock Mantle](https://docs.anthropic.com/en/api/claude-on-amazon-bedrock)
- [Claude model documentation](https://docs.anthropic.com/en/docs/about-claude/models)